In [9]:
from fractions import Fraction
from typing import Tuple, Union
import numpy as np
import pandas as pd
from sympy import Matrix
from IPython.display import display

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.float_format', '{:.15f}'.format)  # Ép hiển thị dạng thập phân chuẩn
pd.set_option('display.width', 300)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp lặp Jacobi giải hệ tuyến tính

# Thuật toán
**Input**: 
- $A_{n \times n}$ và $B_{n \times m}$
- Sai số tuyệt đối $\varepsilon$ hoặc sai số tương đối $\eta$

**Output**: Nghiệm xấp xỉ $X^*$ của hệ phương trình

1. Tính $q=\max\limits_{i=\overline{1,n}}\displaystyle\sum\limits_{\substack{i=1 \\ j \neq i}}^n \dfrac{|a_{ij}|}{|a_{ii}|}$. Nếu $q<1$ thì $p=\infty, \lambda=1$ và qua bước 3
2. Tính $q=\max\limits_{j=\overline{1,n}}\displaystyle\sum\limits_{\substack{j=1 \\ i \neq j}}^n \dfrac{|a_{ij}|}{|a_{jj}|}$. Nếu $q \geq 1 \to$ dừng chương trình, trái lại  $p=1, \lambda=\dfrac{\max|a_{ii}|}{\min|a_{ii}|}$
3. Khởi tạo $T=\text{diag}\left(\dfrac{1}{a_{11}}, \dfrac{1}{a_{22}},\ldots,\dfrac{1}{a_{nn}}\right)$

4. Tính $C=I-TA$ và $D=TB$
5. Khởi tạo $X_0=[0]_{n \times m}$, khởi tạo $k=1$
6. Lặp:
    - Tính $X_{k} = CX_{k-1}+D$ 
    - Tính $\Delta_k = \|X_{k} - X_{k-1}\|_p$ hoặc $\delta_k=\dfrac{\|X_{k}-X_{k-1}\|_p}{\|X_{k}\|_p}$ 

    Lặp cho đến khi $\Delta_k \leq \dfrac{1-q}{\lambda q}\varepsilon$ hoặc $\delta_k \leq \dfrac{1-q}{\lambda q}\eta$
7. In ra $X^*$ là nghiệm xấp xỉ của hệ phương trình


# Thuật toán đánh giá sai số tiên nghiệm (Jacobi)

**Input**: 
- $A_{n \times n}$ và $B_{n \times m}$
- Sai số tuyệt đối $\varepsilon$ hoặc sai số tương đối $\eta$

**Output**: Nghiệm xấp xỉ $X^*$ của hệ phương trình

1. Tính $q=\max\limits_{i=\overline{1,n}}\displaystyle\sum\limits_{\substack{i=1 \\ j \neq i}}^n \dfrac{|a_{ij}|}{|a_{ii}|}$. Nếu $q<1$ thì $p=\infty, \lambda=1$ và qua bước 3.
2. Tính $q=\max\limits_{j=\overline{1,n}}\displaystyle\sum\limits_{\substack{j=1 \\ i \neq j}}^n \dfrac{|a_{ij}|}{|a_{jj}|}$. Nếu $q \ge 1 \to$ dừng chương trình, trái lại $p=1, \lambda=\dfrac{\max|a_{ii}|}{\min|a_{ii}|}$.
3. Khởi tạo $T=\text{diag}\left(\dfrac{1}{a_{11}}, \dfrac{1}{a_{22}},\ldots,\dfrac{1}{a_{nn}}\right)$
4. Tính $C=I-TA$ và $D=TB$.
5. Khởi tạo $X_0=[0]_{n \times m}$. Tính $X_1 = C X_0 + D$. 
6. Đặt $\Delta_1 = \|X_1 - X_0\|_p$. Nếu $\Delta_1 = 0 \Rightarrow$ Trả về $X^* = X_0$ và dừng chương trình.
7. Thiết lập khoảng sai số yêu cầu $\delta$:
    - Nếu dùng sai số tuyệt đối: $\delta = \varepsilon \dfrac{1 - q}{\lambda q}$.
    - Nếu dùng sai số tương đối: $\delta = \eta \|X_1\|_p \dfrac{1 - q}{\lambda q}$.
8. Tính số bước lặp tiên nghiệm $N$:
    - Nếu $\Delta_1 \le \delta \Rightarrow$ Gán $N = 1$.
    - Trái lại $\Rightarrow$ Gán $N = \left\lceil \dfrac{\ln(\delta / \Delta_1)}{\ln(q)} \right\rceil$.
9. Lặp $k$ từ $2$ đến $N$:
    - Tính $X_k = C X_{k-1} + D$.
10. In ra $X^* = X_N$ là nghiệm xấp xỉ của hệ phương trình.

In [10]:
def input_matrix(filename, convert_fractions=False):
    """
    Reads a matrix from a text file and returns it as a NumPy array.
    Supports fractional entries if present.
    """
    matrix = []

    with open(filename, 'r') as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue

            row = []
            for token in tokens:
                if '/' in token:
                    # convert fractions if any includes fraction sign
                    val = Fraction(token)
                    row.append(float(val) if convert_fractions else val)
                else:
                    # parse as float directly
                    row.append(float(token))

            matrix.append(row)
            
    dtype = float if convert_fractions else object
    return np.array(matrix, dtype=dtype)

In [11]:
def output_matrix(X: np.ndarray, precision: int = 15):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    
    Parameters:
    - X: np.ndarray, 1D or 2D array.
    - precision: number of decimal places to round floats to.
    """
    # Render natively as a LaTeX matrix in Jupyter
    # Ép kiểu về float để tránh lỗi np.round với mảng dtype=object
    X_float = np.array(X, dtype=float)
    display(Matrix(np.round(X_float, precision)))

In [12]:
def check_dominance(A: np.ndarray) -> int:
    """
    Returns:
      1 if A is row-dominant only,
      2 if A is column-dominant only,
      3 if both,
      0 if neither.
    """
    n = A.shape[0]
    row_dom = all(abs(A[i,i]) > np.sum(np.abs(A[i,:])) - abs(A[i,i]) for i in range(n))
    col_dom = all(abs(A[j,j]) > np.sum(np.abs(A[:,j])) - abs(A[j,j]) for j in range(n))
    if row_dom and col_dom:
        return 3
    if row_dom:
        return 1
    if col_dom:
        return 2
    return 0

domi_dict = {0: 'Không chéo trội', 1: 'Chéo trội hàng', 2: 'Chéo trội cột', 3: 'Chéo trội cả hàng và cột'}

In [13]:
def convert_to_iteration_row(A: np.ndarray, B: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Converts Ax = B into x_new = C x + D for row-dominant case."""
    T = np.diag(1.0 / np.diag(A))
    n = A.shape[0]
    C = np.eye(n) - T.dot(A)
    D = T.dot(B)
    return C, D

def convert_to_iteration_col(A: np.ndarray, B: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Prepares y_{k+1} = C y_k + D for column-dominant case."""
    T = np.diag(1.0 / np.diag(A))
    n = A.shape[0]
    C = np.eye(n) - A.dot(T)
    D = B.copy()
    return T, C, D


In [14]:
def jacobi_iteration(
    A: np.ndarray,
    B: np.ndarray,
    x0,
    domiType: int,
    eps=None,
    eta=None,
    mode='posteriori',
    max_iterations=1000
):
    if (eps is None) == (eta is None):
        raise ValueError("Specify exactly one of eps (exact) or eta (relative)")

    is_matrix = True if np.array(x0).ndim == 2 and np.array(x0).shape[1] > 1 else False

    if domiType in (1, 3):
        C, D = convert_to_iteration_row(A, B)
        
        print("\nMa trận lặp C:")
        output_matrix(C)
        print("\nVector/Ma trận lặp D:")
        output_matrix(D)
        
        if is_matrix:
            vec_norm = lambda x: np.linalg.norm(x, ord=np.inf)
        else:
            vec_norm = lambda x: np.max(np.abs(x))
            
        q = np.max(np.sum(np.abs(C), axis=1))
        alpha = 1.0
        
        tol = (eps if eps is not None else eta) * (1 - q) / (alpha * q)
        print(f"Sử dụng logic chéo trội hàng. q: {q:.7f}, Ngưỡng sai số (tol): {tol:.7f}")
        print(f"Chế độ lặp: {mode.upper()}")

        x_old = np.array(x0, dtype=float)
        
        if mode == 'apriori':
            x1 = C.dot(x_old) + D
            diff_1 = vec_norm(x1 - x_old)
            if diff_1 == 0:
                print("\n==> X_1 = X_0, nghiệm đã hội tụ ngay từ đầu.")
                max_iterations = 0
            else:
                target_diff = tol if eps is not None else tol * vec_norm(x1) # rough approximation for eta
                if diff_1 <= target_diff:
                    max_iterations = 1
                else:
                    n_steps = np.ceil(np.log(target_diff / diff_1) / np.log(q))
                    max_iterations = int(n_steps) if n_steps > 0 else 0
                print(f"Số bước lặp tiên nghiệm: N = {max_iterations}")

        if not is_matrix:
            x_old = x_old.flatten()
            history = [x_old.copy()]
            errors = [float('nan')]

            for k in range(1, max_iterations + 1):
                x_new = C.dot(x_old) + D
                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                history.append(x_new.copy())
                errors.append(float(err))
                x_old = x_new
                
                if mode == 'posteriori' and tol is not None and err <= tol:
                    break
            else:
                if mode == 'posteriori':
                    print(f"Cảnh báo: Không hội tụ sau {max_iterations} vòng lặp.")
                else:
                    print(f"Đã hoàn thành {max_iterations} vòng lặp tiên nghiệm.")
            
            n = len(x_new)
            import pandas as pd
            df = pd.DataFrame(history, columns=[f"x{i+1}" for i in range(n)])
            df["error"] = errors
            df.index.name = "k"
            df_T = df.T
            df_T.index.name = "X_k"
            return df_T, x_new.reshape(-1, 1)
        else:
            print("X_0 =")
            output_matrix(x_old)
            for k in range(1, max_iterations + 1):
                x_new = C.dot(x_old) + D
                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                print(f"\n--- Vòng lặp k = {k} ---")
                print(f"X_{k} =")
                output_matrix(x_new)
                err_str = f"{err:.4e}" if err < 1e-4 else f"{err:.6f}"
                print(f"Sai số Delta_{k} = {err_str}")
                
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    print(f"\n==> Hội tụ sau {k} bước lặp (sai số <= {tol:.4e}).")
                    break
            else:
                if mode == 'posteriori':
                    print(f"\n==> Đạt giới hạn lặp tối đa {max_iterations} bước.")
                else:
                    print(f"\n==> Đã hoàn thành {max_iterations} bước lặp theo công thức tiên nghiệm.")
            return None, x_old

    elif domiType == 2:
        C, D = convert_to_iteration_row(A, B)
        _, C_q, _ = convert_to_iteration_col(A, B)

        print("\nMa trận lặp C:")
        output_matrix(C)
        print("\nVector/Ma trận lặp D:")
        output_matrix(D)

        if is_matrix:
            vec_norm = lambda x: np.linalg.norm(x, ord=1) 
        else:
            vec_norm = lambda x: np.sum(np.abs(x))
            
        q = np.max(np.sum(np.abs(C_q), axis=0))
        diag_vals = np.abs(np.diag(A))
        alpha = np.max(diag_vals) / np.min(diag_vals)

        tol = (eps if eps is not None else eta) * (1 - q) / (alpha * q)
        print(f"Sử dụng logic chéo trội cột. q: {q:.7f}, α: {alpha:.7f}, Ngưỡng sai số (tol): {tol:.7f}")
        print(f"Chế độ lặp: {mode.upper()}")

        x_old = np.array(x0, dtype=float)
        
        if mode == 'apriori':
            x1 = C.dot(x_old) + D
            diff_1 = vec_norm(x1 - x_old)
            if diff_1 == 0:
                print("\n==> X_1 = X_0, nghiệm đã hội tụ ngay từ đầu.")
                max_iterations = 0
            else:
                target_diff = tol if eps is not None else tol * vec_norm(x1)
                if diff_1 <= target_diff:
                    max_iterations = 1
                else:
                    n_steps = np.ceil(np.log(target_diff / diff_1) / np.log(q))
                    max_iterations = int(n_steps) if n_steps > 0 else 0
                print(f"Số bước lặp tiên nghiệm: N = {max_iterations}")
        
        if not is_matrix:
            x_old = x_old.flatten()
            history = [x_old.copy()]
            errors = [float('nan')]
            for k in range(1, max_iterations + 1):
                x_new = C.dot(x_old) + D
                
                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                history.append(x_new.copy())
                errors.append(float(err))
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    break
            else:
                if mode == 'posteriori':
                    print(f"Cảnh báo: Không hội tụ sau {max_iterations} vòng lặp.")
                else:
                    print(f"Đã hoàn thành {max_iterations} vòng lặp tiên nghiệm.")
            
            n = len(x_new)
            import pandas as pd
            df = pd.DataFrame(history, columns=[f"x{i+1}" for i in range(n)])
            df["error"] = errors
            df.index.name = "k"
            df_T = df.T
            df_T.index.name = "X_k"
            return df_T, x_new.reshape(-1, 1)
        else:
            print("X_0 =")
            output_matrix(x_old)
            for k in range(1, max_iterations + 1):
                x_new = C.dot(x_old) + D
                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                print(f"\n--- Vòng lặp k = {k} ---")
                print(f"X_{k} =")
                output_matrix(x_new)
                err_str = f"{err:.4e}" if err < 1e-4 else f"{err:.6f}"
                print(f"Sai số Delta_{k} = {err_str}")
                
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    print(f"\n==> Hội tụ sau {k} bước lặp (sai số <= {tol:.4e}).")
                    break
            else:
                if mode == 'posteriori':
                    print(f"\n==> Đạt giới hạn lặp tối đa {max_iterations} bước.")
                else:
                    print(f"\n==> Đã hoàn thành {max_iterations} bước lặp theo công thức tiên nghiệm.")
            return None, x_old
            
    else:
        raise ValueError("Ma trận không chéo trội, phương pháp Jacobi có thể không hội tụ.")

def solve_jacobi_system(A, B, eps=1e-6, eta=None, mode='posteriori', max_iterations=1000):
    domiType = check_dominance(A)
    if domiType == 0:
        print("Lỗi: Ma trận không chéo trội. Thuật toán tự động kết thúc.")
        return None

    if B.ndim == 1 or B.shape[1] == 1:
        B_mat = B.flatten()
        x0 = np.zeros_like(B_mat)
        df_history, X_final = jacobi_iteration(A, B_mat, x0, domiType, eps, eta, mode, max_iterations)
        display(df_history)
        print("\nNghiệm gần đúng:")
        output_matrix(X_final)
        return X_final
    else:
        B_mat = B
        x0 = np.zeros_like(B_mat)
        print("\n--- GIẢI CHO HỆ VỚI VẾ PHẢI LÀ MA TRẬN ---")
        _, X_final = jacobi_iteration(A, B_mat, x0, domiType, eps, eta, mode, max_iterations)
        print("\n=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:")
        print("Hệ có nghiệm X =")
        output_matrix(X_final)
        return X_final


### Kết quả

In [15]:
# Test với ma trận JCB_input_A2 và JCB_input_B2
A = input_matrix('A34.txt', convert_fractions=False)
B = input_matrix('B34.txt', convert_fractions=False) # B is now a matrix

print("\nMa trận A:")
output_matrix(A)
print("\nKiểm tra chéo trội của A:", domi_dict[check_dominance(A)])
print("\nMa trận B:")
output_matrix(B)

# Giải hệ
eps = 1e-6
X = solve_jacobi_system(A, B, eps=eps)


Ma trận A:


Matrix([
[211.0,  22.0, -13.0,  24.0,  15.0, -26.0,  17.0,  28.0],
[ 22.0, 433.0,  24.0,  35.0,  26.0,  37.0,  28.0, -39.0],
[ 33.0, -24.0, 235.0, -26.0,  37.0,  28.0, -39.0,  20.0],
[ 14.0,  45.0,  26.0, 247.0,  38.0,  49.0,  40.0, -41.0],
[-55.0,  16.0,  57.0,  28.0, 259.0,  30.0, -51.0,  42.0],
[ 46.0,  27.0, -48.0,  39.0,  40.0, 261.0,  42.0,  73.0],
[ 27.0, -58.0,  29.0,  70.0, -21.0,  42.0, 223.0,  34.0],
[ 38.0,  59.0,  60.0, -71.0,  82.0, -93.0,  24.0, 215.0]])


Kiểm tra chéo trội của A: Không chéo trội

Ma trận B:


Matrix([
[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]])

Lỗi: Ma trận không chéo trội. Thuật toán tự động kết thúc.


In [16]:
# Test với ma trận JCB_input_A1 và JCB_input_B1
A = input_matrix('JCB_input_A1.txt', convert_fractions=True)
B = input_matrix('JCB_input_B1.txt', convert_fractions=True)

print("\nMa trận A:")
output_matrix(A)
print("\nKiểm tra chéo trội của A:", domi_dict[check_dominance(A)])
print("\nMa trận B:")
output_matrix(B)

# Giải hệ
eps = 1e-6
X = solve_jacobi_system(A, B, eps=eps)


Ma trận A:


Matrix([
[14.4,  0.2,  0.6, -0.1,  0.9,  0.4,  0.9],
[ 0.4, 16.3, -0.4, -0.7, -0.5,  0.1, -0.6],
[ 0.7,  0.9, 18.6, -0.3,  0.6, -0.2,  0.4],
[-0.3, -1.0, -0.1, 19.7, -0.7,  0.3, -0.6],
[ 0.6, -0.1, -1.0, -0.6, 20.6,  0.3, -0.8],
[ 0.4, -0.2, -0.7,  0.5, -0.9, 22.9,  0.2],
[-1.0, -0.1,  0.5, -0.5,  0.2,  0.3, 23.9]])


Kiểm tra chéo trội của A: Chéo trội cả hàng và cột

Ma trận B:


Matrix([
[-10.0],
[ -3.0],
[ -7.0],
[  6.0],
[ -4.0],
[  1.0],
[ -7.0]])


Ma trận lặp C:


...

Matrix([
[-0.694444444444444],
[-0.184049079754601],
[-0.376344086021505],
[ 0.304568527918782],
[-0.194174757281553],
[ 0.043668122270742],
[-0.292887029288703]])

Sử dụng logic chéo trội hàng. q: 0.2152778, Ngưỡng sai số (tol): 0.0000036
Chế độ lặp: POSTERIORI


k,0,1,2,3,4,5,6
X_k,,,,,,,
x1,0.000000000000000,-0.694444444444444,-0.644863786155511,-0.646002302313993,-0.645912928903010,-0.645913255625913,-0.645913180199306
x2,0.000000000000000,-0.184049079754601,-0.180168607087653,-0.182250049765581,-0.182250856057769,-0.182257542858493,-0.182257596526595
x3,0.000000000000000,-0.376344086021505,-0.323359186462126,-0.325788584788586,-0.325609060378150,-0.325613518101865,-0.325613180046732
x4,0.000000000000000,0.304568527918782,0.266255238271999,0.267154178609740,0.266991981486837,0.266992915961437,0.266992538871890
x5,0.000000000000000,-0.194174757281553,-0.196250061684040,-0.196597423456670,-0.196676537347664,-0.196673751900165,-0.196674081716391
x6,0.000000000000000,0.043668122270742,0.030963448120382,0.032632576503658,0.032523621397579,0.032528014605771,0.032527926799199
x7,0.000000000000000,-0.292887029288703,-0.307391570321116,-0.307033993999854,-0.307038753717621,-0.307040136925776,-0.307040144220996
error,NaN,0.694444444444444,0.052984899559379,0.002429398326460,0.000179524410437,0.000006686800724,0.000000377089547



Nghiệm gần đúng:


Matrix([
[-0.645913180199306],
[-0.182257596526595],
[-0.325613180046732],
[  0.26699253887189],
[-0.196674081716391],
[ 0.032527926799199],
[-0.307040144220996]])

In [17]:
# Test với ma trận JCB_input_A1 và JCB_input_B1
A = input_matrix('JCB_input_A_col.txt', convert_fractions=True)
B = input_matrix('JCB_input_B_col.txt', convert_fractions=True)

print("\nMa trận A:")
output_matrix(A)
print("\nKiểm tra chéo trội của A:", domi_dict[check_dominance(A)])
print("\nMa trận B:")
output_matrix(B)

# Giải hệ
eps = 1e-6
X = solve_jacobi_system(A, B, eps=eps)


Ma trận A:


Matrix([
[2.0, 3.0, 2.0],
[1.0, 5.0, 1.0],
[0.0, 1.0, 4.0]])


Kiểm tra chéo trội của A: Chéo trội cột

Ma trận B:


Matrix([
[7.0],
[7.0],
[5.0]])


Ma trận lặp C:


...

Matrix([
[ 3.5],
[ 1.4],
[1.25]])

Sử dụng logic chéo trội cột. q: 0.8000000, α: 2.5000000, Ngưỡng sai số (tol): 0.0000001
Chế độ lặp: POSTERIORI


k,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44
X_k,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
x1,0.000000000000000,3.500000000000000,0.150000000000000,1.925000000000000,0.577500000000000,1.366250000000000,0.805874999999999,1.149312500000000,0.913743750000000,1.061965625000000,0.962344687500000,1.026000781250000,0.983722359375000,1.010983039062500,0.993002786718750,1.004657945703125,0.997001823398437,1.001980141660156,0.998717740904297,1.000842958411133,0.999452202233496,1.000359148398681,0.999766122861167,1.000153091827864,0.999900185581474,1.000065275996694,0.999957410362123,1.000027837319769,0.999981829826909,1.000011872543813,0.999992248573429,1.000005063898989,0.999996693373509,1.000002159935975,0.999998589485779,1.000000921308915,0.999999398323223,1.000000392983831,0.999999743347682,1.000000167628180,0.999999890522497,1.000000071502479,0.999999953301465,1.000000030499742,0.999999980080389
x2,0.000000000000000,1.400000000000000,0.450000000000000,1.190000000000000,0.787500000000000,1.094000000000000,0.916125000000000,1.043525000000000,0.965943750000000,1.019427500000000,0.985904062500000,1.008502437500000,0.994095046875000,1.003680650000000,0.997508144531250,1.001583475156250,0.998943818085938,1.000678809078125,0.999551162572266,1.000290392273047,0.999808966446387,1.000124079166953,0.999918618642583,1.000052979386114,0.999965312566557,1.000022611853011,0.999985210428989,1.000009648520226,0.999993693057496,1.000004116460630,0.999997310144112,1.000001756108346,0.999998852727408,1.000000749130716,0.999999510649176,1.000000319559380,0.999999791270676,1.000000136313325,0.999999910966768,1.000000058146130,0.999999962022702,1.000000024802807,0.999999983800639,1.000000010579847,0.999999993090084
x3,0.000000000000000,1.250000000000000,0.900000000000000,1.137500000000000,0.952500000000000,1.053125000000000,0.976500000000000,1.020968750000000,0.989118750000000,1.008514062500000,0.995143125000000,1.003523984375000,0.997874390625000,1.001476238281250,0.999079837500000,1.000622963867188,0.999604131210937,1.000264045478516,0.999830297730469,1.000112209356934,0.999927401931738,1.000047758388403,0.999968980208262,1.000020345339354,0.999986755153471,1.000008671858361,0.999994347036747,1.000003697392753,0.999997587869944,1.000001576735626,0.999998970884843,1.000000672463972,0.999999560972914,1.000000286818148,0.999999812717321,1.000000122337706,0.999999920110155,1.000000052182331,0.999999965921669,1.000000022258308,0.999999985463468,1.000000009494324,0.999999993799298,1.000000004049840,0.999999997355038
error,NaN,6.150000000000000,4.650000000000000,2.752500000000000,1.935000000000000,1.195875000000001,0.814875000000001,0.515306250000001,0.345000000000000,0.221100937500000,0.146515312500000,0.094635328125000,0.062335406250000,0.040448130468750,0.026549158593750,0.017273615976562,0.011314612031250,0.007373223521484,0.004823795009766,0.003146358834082,0.002056989429493,0.001342415342418,0.000877264242026,0.000572694841320,0.000374163251830,0.000244306406563,0.000159591880206,0.000104215404888,0.000068072478400,0.000044454985721,0.000029036137685,0.000018962868923,0.000012385397476,0.000008088811007,0.000005283032563,0.000003450353727,0.000002253501948,0.000001471775433,0.000000961243369,0.000000627796499,0.000000410023951,0.000000267790943,0.000000174898208,0.000000114228027,0.000000074603919



Nghiệm gần đúng:


Matrix([
[0.999999980080389],
[0.999999993090084],
[0.999999997355038]])